# 🛠️ Notebook 3: Build a Tiny Metrics Collector

In this notebook we build a **mini Prometheus** — a tiny in-memory metrics collector with counters, gauges, and histograms. Then we add **labels** (so we can slice by endpoint), see how labels can blow up your memory (**cardinality**), and finally implement a **bucketed histogram** the way real monitoring systems actually do it.

## Learning objectives
- Implement counters, gauges, and bucketed histograms in <100 lines.
- Add **labels** so a single metric name can carry multiple time series.
- Recognize and avoid the **cardinality explosion** anti-pattern.
- Compute approximate percentiles from histogram buckets (the *Prometheus* way) and contrast with exact percentiles from raw samples (the *teaching* way).
- Plot a latency histogram with matplotlib.

## Step 1 — A tiny `Registry` with counters, gauges, and raw-sample histograms

For pedagogy we start by storing **every sample** so we can compute exact percentiles. We'll see why real systems don't do this in Step 3.

In [ ]:
import time, random, bisect
from collections import defaultdict

class Registry:
    """Tiny metrics registry. Labels are stored as a tuple of (key, value) pairs."""
    def __init__(self):
        self._counters: dict[tuple, float] = defaultdict(float)
        self._gauges:   dict[tuple, float] = {}
        self._samples:  dict[tuple, list[float]] = defaultdict(list)

    @staticmethod
    def _key(name, labels):
        # Sorted tuple so {a=1,b=2} and {b=2,a=1} share the same series.
        return (name, tuple(sorted((labels or {}).items())))

    # --- counters ---
    def inc(self, name, by=1.0, **labels):
        self._counters[self._key(name, labels)] += by

    # --- gauges ---
    def set(self, name, v, **labels):
        self._gauges[self._key(name, labels)] = v

    # --- histograms (raw samples — easy to reason about, expensive in production) ---
    def observe(self, name, v, **labels):
        self._samples[self._key(name, labels)].append(v)

    def percentile(self, name, p, **labels):
        s = sorted(self._samples.get(self._key(name, labels), []))
        if not s:
            return None
        k = max(0, min(len(s) - 1, int(round(p * (len(s) - 1)))))
        return s[k]

    # --- public accessors (don't poke private attrs from outside) ---
    def samples(self, name, **labels):
        return list(self._samples.get(self._key(name, labels), []))

    def series_count(self):
        return len(self._counters) + len(self._gauges) + len(self._samples)

    def render(self):
        """Prometheus-ish text format."""
        def fmt_labels(lbls):
            if not lbls:
                return ""
            return "{" + ",".join(f'{k}="{v}"' for k, v in lbls) + "}"

        lines = []
        seen_types = set()
        def maybe_type(name, kind):
            if name not in seen_types:
                lines.append(f"# TYPE {name} {kind}")
                seen_types.add(name)

        for (name, lbls), v in self._counters.items():
            maybe_type(name, "counter")
            lines.append(f"{name}{fmt_labels(lbls)} {v}")
        for (name, lbls), v in self._gauges.items():
            maybe_type(name, "gauge")
            lines.append(f"{name}{fmt_labels(lbls)} {v}")
        for (name, lbls) in self._samples:
            maybe_type(name, "summary")
            for p in (0.5, 0.95, 0.99):
                v = self.percentile(name, p, **dict(lbls))
                lines.append(f'{name}{{quantile="{p}"{("," + ",".join(k+chr(61)+chr(34)+v2+chr(34) for k,v2 in lbls)) if lbls else ""}}} {v:.3f}')
        return "\n".join(lines)


## Step 2 — Use it: requests with **labels**

A single metric name (`http_requests_total`) becomes many *time series* once you add labels — one per unique label combination. That's incredibly powerful (you can break down by endpoint, status, method, region…) but also dangerous, as we'll see.

In [ ]:
m = Registry()
random.seed(0)

endpoints = ["/checkout", "/cart", "/search"]
for _ in range(5000):
    ep = random.choice(endpoints)
    # /checkout is intentionally slower
    base = 200 if ep == "/checkout" else 60
    latency = random.lognormvariate(0, 0.6) * base
    status = "200" if random.random() > 0.01 else "500"
    m.inc("http_requests_total", endpoint=ep, status=status)
    m.observe("http_latency_ms", latency, endpoint=ep)

m.set("memory_in_use_mb", 412)

print("Total time series stored:", m.series_count())
print()
for ep in endpoints:
    p95 = m.percentile("http_latency_ms", 0.95, endpoint=ep)
    print(f"  {ep:<10} p95 = {p95:6.1f} ms")

## Step 3 — ⚠️ The cardinality trap

Each unique combination of `(metric_name, labels)` is a separate **time series** that the monitoring backend must store, index, and query. Beginners often add a label like `user_id` or `request_id` thinking "more detail is better!" — and accidentally create *millions* of series.

> **Rule of thumb:** label values must come from a **small, bounded** set (`endpoint`, `status_code`, `region`, `method`). Never use unbounded values like user IDs, email addresses, request IDs, URLs with parameters, etc.

Let's see the difference.

In [ ]:
bad = Registry()
for i in range(2000):
    # ❌ user_id is unbounded -> one series per user.
    bad.inc("http_requests_total", endpoint="/checkout", user_id=f"user-{i}")

good = Registry()
for i in range(2000):
    # ✅ Bucket the user into a small set of plan tiers.
    plan = random.choice(["free", "pro", "enterprise"])
    good.inc("http_requests_total", endpoint="/checkout", plan=plan)

print(f"❌ bad  registry: {bad.series_count():>5} series  (will OOM in production)")
print(f"✅ good registry: {good.series_count():>5} series  (fine)")

For *per-user* breakdowns, use **logs or traces** — they're built for high-cardinality data. Metrics are for low-cardinality aggregates.

## Step 4 — Bucketed histograms (the *real* Prometheus way)

Storing every raw sample (Step 1) is great for teaching but doesn't scale: at 100k req/s your samples list explodes. Real systems use **bucketed histograms**: pre-defined upper bounds (`le` = "less than or equal to"), and you just count how many observations fell into each bucket.

Important Prometheus details:
- Buckets are **cumulative** — `le=0.5` includes everything `≤ 0.5` *and* everything in smaller buckets.
- Each bucket is a counter; you also keep `_sum` and `_count`.
- Percentiles from buckets are **approximations** (Prometheus's `histogram_quantile` does linear interpolation inside the right bucket).

Don't expect bucket-based p95 to match raw-sample p95 exactly — that's the trade-off you make to save memory.

In [ ]:
class BucketedHistogram:
    """A Prometheus-style histogram: cumulative bucket counts + _sum + _count."""
    def __init__(self, buckets):
        # Always include +inf as the last bucket so every observation lands somewhere.
        self.bounds = sorted(buckets) + [float("inf")]
        self.counts = [0] * len(self.bounds)   # CUMULATIVE counts
        self.sum = 0.0
        self.count = 0

    def observe(self, v):
        self.sum += v
        self.count += 1
        # Increment every bucket whose upper bound is >= v (cumulative).
        idx = bisect.bisect_left(self.bounds, v)
        for i in range(idx, len(self.bounds)):
            self.counts[i] += 1

    def quantile(self, q):
        """Approximate quantile via linear interpolation inside the matching bucket."""
        if self.count == 0:
            return None
        target = q * self.count
        prev_count, prev_bound = 0, 0.0
        for c, b in zip(self.counts, self.bounds):
            if c >= target:
                if b == float("inf"):
                    return prev_bound
                bucket_size = c - prev_count
                if bucket_size == 0:
                    return b
                # Linear interpolation between prev_bound and b.
                frac = (target - prev_count) / bucket_size
                return prev_bound + frac * (b - prev_bound)
            prev_count, prev_bound = c, b
        return self.bounds[-2]

    def render(self, name):
        out = [f"# TYPE {name} histogram"]
        for c, b in zip(self.counts, self.bounds):
            le = "+Inf" if b == float("inf") else f"{b:g}"
            out.append(f'{name}_bucket{{le="{le}"}} {c}')
        out.append(f"{name}_sum {self.sum:.3f}")
        out.append(f"{name}_count {self.count}")
        return "\n".join(out)


# Buckets in milliseconds — pick them to cover your expected range.
hist = BucketedHistogram([5, 10, 25, 50, 100, 250, 500, 1000, 2500])

raw_samples = []
random.seed(0)
for _ in range(5000):
    latency = random.lognormvariate(4, 0.6)
    hist.observe(latency)
    raw_samples.append(latency)

print(hist.render("http_latency_ms"))
print()
print("Approx (bucketed) vs Exact (raw) percentiles:")
for q in (0.5, 0.95, 0.99):
    raw_sorted = sorted(raw_samples)
    exact = raw_sorted[int(q * (len(raw_sorted) - 1))]
    approx = hist.quantile(q)
    print(f"  p{int(q*100)}: bucketed ≈ {approx:7.1f} ms   raw = {exact:7.1f} ms")

Notice the bucketed values are **close but not equal** to the exact percentiles. That's the trade-off:

|  | Memory | Accuracy | Aggregation across servers |
|---|---|---|---|
| Raw samples | 💸 grows forever | exact | hard |
| Bucketed | 💚 fixed (one int per bucket) | approximate | easy (just sum buckets) |

Production monitoring systems pick bucketed because (a) memory is bounded and (b) you can sum the bucket counters from 100 servers and *still* compute a global percentile.

## Step 5 — Visualize the latency distribution

In [ ]:
import matplotlib.pyplot as plt

samples = m.samples("http_latency_ms", endpoint="/checkout")
plt.figure(figsize=(8, 4))
plt.hist(samples, bins=60, color="steelblue", edgecolor="white")
for p, color in [(0.5, "green"), (0.95, "orange"), (0.99, "red")]:
    v = m.percentile("http_latency_ms", p, endpoint="/checkout")
    plt.axvline(v, color=color, linestyle="--", label=f"p{int(p*100)} = {v:.0f} ms")
plt.title("HTTP request latency — /checkout")
plt.xlabel("latency (ms)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Recap

A real metrics system (Prometheus, Datadog, OpenTelemetry…) is doing roughly the same thing as this notebook — just with:

- **bucketed histograms** instead of full sample arrays (saves memory, aggregates across hosts),
- a **sliding time window** so you see *recent* values, not lifetime totals,
- **pull or push over the network** with a text/binary exposition format,
- **labels** for slicing — kept low-cardinality on purpose.

Watch out for the **cardinality trap**: high-cardinality fields (user IDs, request IDs, full URLs) belong in **logs** or **traces**, not in metric labels.

In the final notebook we'll **debug a fake outage** using all three pillars together.